# PWR Rod Bundle

### This is a computation of the 13-2 problem from the ANL benchmark book ![](./pwr_rod_bundle.png)

In [1]:
import inspect

import numpy as np
from IPython.display import Code
from numba.np.unsafe.ndarray import *  # noqa: F403

from dorban.examples import anl_13_pwr_rod_bundle
from dorban.settings import FDSettings
from dorban.solve_equation import solve_k
from dorban.system import mesh_refinement

## The problem
The geometry, the materials, their arrangement in the core and the cores are defined in `dorban.examples.anl_13_pwr_rod_bundle`, which the benchmark tests use too:

In [ ]:
Code(inspect.getsource(anl_13_pwr_rod_bundle), language="python")

## Definitions of the current calculator and of the core

In [ ]:
system = anl_13_pwr_rod_bundle.core()

## Refinemnt and solution of the diffusion equation, reference multiplication factor is 1.0855

In [6]:
sub_cells = anl_13_pwr_rod_bundle.FD_SUB_CELLS
split = system.geometry.uniform_split(sub_cells)
settings = FDSettings(split=split)
k, flux = solve_k(system, settings)

In [7]:
print(f"k = {k:.4f}, reference {anl_13_pwr_rod_bundle.REFERENCE_K}")

## Computation of the ppf, reference ppf is 1.418.

In [8]:
refined_system = mesh_refinement(system, split)
k, fine_flux = solve_k(
    refined_system, FDSettings(split=refined_system.geometry.uniform_split(1))
)
ppf = anl_13_pwr_rod_bundle.power_peaking_factor(refined_system, fine_flux)

In [9]:
print(f"power peaking factor = {ppf:.3f}, reference {anl_13_pwr_rod_bundle.REFERENCE_PPF}")

In [10]:
reference = anl_13_pwr_rod_bundle.reference_flux()
reorded_flux = anl_13_pwr_rod_bundle.reorder_flux(flux)
deviation = np.linalg.norm((reorded_flux - reference) / reference) / len(reorded_flux - reference)
print(f"norm of the relative differences of the flux from the reference, divided by their number: {deviation:.2e}")

## Compare NEM to finite differences

In [11]:
nem_system = anl_13_pwr_rod_bundle.nem_core()

In [12]:
from dorban.nem.solve_nem import NEMSettings

sub_cells = anl_13_pwr_rod_bundle.NEM_SUB_CELLS
split = nem_system.geometry.uniform_split(sub_cells)
settings = NEMSettings(split=split)
k_nem, flux_nem = solve_k(nem_system, settings)

In [13]:
print(f"NEM k = {k_nem:.5f}, finite differences k = {k:.5f}")

In [14]:
normalized_flux=flux/np.sum(flux)
normalized_nem=flux_nem/np.sum(flux_nem)

In [15]:
print(f"largest relative difference of the flux, NEM against finite differences: {np.max(np.abs(normalized_flux-normalized_nem)/normalized_flux):.2%}")